# E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [47]:
import torch
import matplotlib.pyplot as plt

words = open('names.txt', 'r').read().splitlines()

In [48]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

### E01: Using counting

In [49]:
N = torch.zeros((27, 27, 27), dtype=torch.int32)

In [50]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        N[ix1, ix2, ix3] += 1

In [51]:
P = (N + 1).float()
P /= P.sum(2, keepdim=True)

In [52]:
# Sampling

g = torch.Generator().manual_seed(2147483647)

for i in range(20):
  out = []
  ix1 = 0   # second-to-last character (start with '.' twice)
  ix2 = 0   # last character
  while True:
    # p = N[ix1, ix2, :].float() + 1
    # p = p / p.sum()
    p = P[ix1][ix2]
    ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix3])
    if ix3 == 0:
      break
    ix1, ix2 = ix2, ix3
  print(''.join(out))

ce.
za.
zogh.
uriana.
kaydnevonimittain.
luwak.
ka.
da.
samiyah.
javer.
gotai.
is.
iselivojkwuthda.
kaley.
maside.
en.
bvgyn.
wynnstihiliven.
tahlasuzusfxx.
leenlen.


In [53]:
# Answer
log_likelihood = 0.0
n = 0

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob = P[ix1, ix2, ix3]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-410414.9688)
nll=tensor(410414.9688)
2.092747449874878


### E01: Using a neural net

In [54]:
xs, ys, zs = [], [], []
# for w in words[:1]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    # print(ch1, ch2, ch3)
    xs.append(ix1)
    ys.append(ix2)
    zs.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)

xs_len = xs.nelement()
ys_len = ys.nelement()

In [55]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27, 27), generator=g, requires_grad=True)

In [56]:
# gradient descent
for k in range(100):
    logits = W[xs, ys, :] # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(xs_len), zs].log().mean()
    print(loss.item())
    
    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()
    
    # update
    W.data += -500 * W.grad

3.723123550415039
3.133915662765503
2.8958654403686523
2.739251136779785
2.6541614532470703
2.5815513134002686
2.5432560443878174
2.484405755996704
2.454169988632202
2.4290549755096436
2.4220492839813232
2.375354051589966
2.3558075428009033
2.339423179626465
2.325371503829956
2.312711715698242
2.303337812423706
2.2976315021514893
2.304532766342163
2.274078607559204
2.2672832012176514
2.264862298965454
2.27486252784729
2.2454142570495605
2.239663600921631
2.2377266883850098
2.2463107109069824
2.2263948917388916
2.2292399406433105
2.2220234870910645
2.233743190765381
2.207505464553833
2.2043542861938477
2.204848051071167
2.216251850128174
2.195674180984497
2.197885036468506
2.1967313289642334
2.211559772491455
2.1830530166625977
2.179579973220825
2.176877498626709
2.1751046180725098
2.1734225749969482
2.175321578979492
2.1785480976104736
2.195000410079956
2.1663036346435547
2.1634438037872314
2.161381244659424
2.160388231277466
2.160107135772705
2.1634645462036133
2.1773695945739746
2.15

In [57]:
ys

tensor([ 5, 13, 13,  ..., 25, 26, 24])

In [58]:
# Sampling

g = torch.Generator().manual_seed(2147483647)

for i in range(20):
  out = []
  ix1 = 0   # second-to-last character (start with '.' twice)
  ix2 = 0   # last character
  while True:
    # p = N[ix1, ix2, :].float() + 1
    # p = p / p.sum()
    logits = W[ix1, ix2, :]
    counts = logits.exp()
    p = counts / counts.sum()
    ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix3])
    if ix3 == 0:
      break
    ix1, ix2 = ix2, ix3
  print(''.join(out))

zexzdfzjglkuria.
prityharlonimittain.
luwan.
ka.
or.
zamiyah.
pathrighton.
is.
iselivojkwu.
oda.
kaley.
zessdguetkaviony.
yobspehlynne.
vtahlas.
zayde.
.
za.
ol.
pen.
.


# E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [98]:
import numpy as np
import torch.nn.functional as F

In [99]:
word_len = len(words)
rng = np.random.default_rng(seed=42)
training_end = int(0.8 * n)
dev_end = int(0.9 * n)
train_idx, dev_idx, test_idx = np.split(
    rng.permutation(word_len),
    [int(.8*word_len), int(.9*word_len)]
)

In [100]:
training_set = [words[i] for i in train_idx]
dev_set = [words[i] for i in dev_idx]
test_set = [words[i] for i in test_idx]

### E02: Bigram

In [101]:
def initialize_set(word_set: list[str]):
    xs, ys = [], []
    for w in word_set:
      chs = ['.'] + list(w) + ['.']
      for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
    return torch.tensor(xs), torch.tensor(ys)

xs_train, ys_train = initialize_set(training_set)
xs_dev, ys_dev = initialize_set(dev_set)
xs_test, ys_test = initialize_set(test_set)

In [102]:
# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [106]:
# gradient descent
for k in range(100):
  xenc = F.one_hot(xs_train, num_classes=27).float()
  logits = xenc @ W
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)
  loss = -probs[torch.arange(xs_train.nelement()), ys_train].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

2.4818239212036133
2.4817984104156494
2.481773853302002
2.4817488193511963
2.481724500656128
2.4817004203796387
2.4816765785217285
2.4816534519195557
2.4816300868988037
2.481607675552368
2.4815852642059326
2.4815633296966553
2.481541395187378
2.4815196990966797
2.4814984798431396
2.481477737426758
2.481456756591797
2.4814364910125732
2.4814162254333496
2.4813966751098633
2.481376886367798
2.4813575744628906
2.4813385009765625
2.4813194274902344
2.4813008308410645
2.4812824726104736
2.481264352798462
2.4812464714050293
2.481228828430176
2.481210947036743
2.481194019317627
2.4811766147613525
2.4811599254608154
2.4811432361602783
2.4811267852783203
2.4811103343963623
2.4810943603515625
2.481078624725342
2.4810631275177
2.4810476303100586
2.481032371520996
2.4810171127319336
2.4810023307800293
2.480987787246704
2.480973243713379
2.4809584617614746
2.4809443950653076
2.4809303283691406
2.4809162616729736
2.4809024333953857
2.480888605117798
2.480875253677368
2.4808621406555176
2.48084902763

In [107]:
def evaluate(set_name: str, xs_tensor, ys_tensor) -> None:
    xenc = F.one_hot(xs_tensor, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(xs_tensor.nelement()), ys_tensor].log().mean() + 0.01*(W**2).mean()
    print(f"{set_name}: {loss.item()}")

evaluate("dev", xs_dev, ys_dev)
evaluate("test", xs_test, ys_test)

dev: 2.4779274463653564
test: 2.4971389770507812


### E02: Trigram

In [115]:
def initialize_set(word_set: list[str]):
    xs, ys, zs = [], [], []
    for w in word_set:
      chs = ['.'] + list(w) + ['.']
      for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)
    return torch.tensor(xs), torch.tensor(ys), torch.tensor(zs)

xs_train, ys_train, zs_train = initialize_set(training_set)
xs_dev, ys_dev, zs_dev = initialize_set(dev_set)
xs_test, ys_test, zs_test = initialize_set(test_set)

In [116]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27, 27), generator=g, requires_grad=True)

In [118]:
# gradient descent
for k in range(100):
    logits = W[xs_train, ys_train, :] # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(xs_train.nelement()), zs_train].log().mean()
    print(loss.item())
    
    # backward pass
    W.grad = None # set to zero the gradient
    loss.backward()
    
    # update
    W.data += -500 * W.grad

2.122709274291992
2.1246562004089355
2.1425790786743164
2.115191698074341
2.11362361907959
2.1131014823913574
2.1145293712615967
2.120598077774048
2.1216320991516113
2.1385035514831543
2.1125800609588623
2.1103875637054443
2.1102259159088135
2.10953950881958
2.109531879425049
2.1091315746307373
2.1104190349578857
2.113882303237915
2.12847638130188
2.1106550693511963
2.1156091690063477
2.1167521476745605
2.133957624435425
2.1076292991638184
2.1062700748443604
2.1055800914764404
2.105682849884033
2.105584144592285
2.108046770095825
2.1130385398864746
2.130330801010132
2.105449676513672
2.104534387588501
2.105945587158203
2.1142501831054688
2.1119649410247803
2.1279826164245605
2.103639841079712
2.1029465198516846
2.104362726211548
2.1127142906188965
2.110454559326172
2.1265599727630615
2.1021978855133057
2.1014902591705322
2.102854013442993
2.1109719276428223
2.109280824661255
2.1256229877471924
2.1008358001708984
2.099956750869751
2.1006810665130615
2.1064724922180176
2.109281063079834


In [119]:
def evaluate(set_name: str, xs_tensor, ys_tensor, zs_tensor) -> None:
    logits = W[xs_tensor, ys_tensor, :]
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(xs_tensor.nelement()), zs_tensor].log().mean()
    print(f"{set_name}: {loss.item()}")

evaluate("dev", xs_dev, ys_dev, zs_dev)
evaluate("test", xs_test, ys_test, zs_test)

dev: 2.1278505325317383
test: 2.1281206607818604


# E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

In [144]:
g = torch.Generator().manual_seed(2147483647)

In [149]:
def train(weights, regularization_strength: float):
    final_loss = None
    for k in range(100):
        logits = weights[xs_train, ys_train, :]
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)
        loss = -probs[torch.arange(xs_train.nelement()), zs_train].log().mean() + regularization_strength*(weights**2).mean()
        final_loss = loss.item()
        
        weights.grad = None
        loss.backward()
        
        weights.data += -500 * weights.grad
    print(f"Final loss for regularization strength {regularization_strength}: {final_loss}")

W1 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W2 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W3 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W4 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W5 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W6 = torch.randn((27, 27, 27), generator=g, requires_grad=True)
W7 = torch.randn((27, 27, 27), generator=g, requires_grad=True)

train(W1, 0.00001)
train(W2, 0.0001)
train(W3, 0.001)
train(W4, 0.01)
train(W5, 0.1)
train(W6, 0.5)
train(W7, 0.9)

Final loss for regularization strength 1e-05: 2.1146655082702637
Final loss for regularization strength 0.0001: 2.126330852508545
Final loss for regularization strength 0.001: 2.119178295135498
Final loss for regularization strength 0.01: 2.1292037963867188
Final loss for regularization strength 0.1: 2.210099697113037
Final loss for regularization strength 0.5: 2.3500001430511475
Final loss for regularization strength 0.9: 2.444209337234497


In [156]:
def evaluate(set_name: str, weights, xs_tensor, ys_tensor, zs_tensor) -> None:
    logits = weights[xs_tensor, ys_tensor, :]
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(xs_tensor.nelement()), zs_tensor].log().mean()
    print(f"{set_name} loss: {loss.item()}")

evaluate("dev", W1, xs_dev, ys_dev, zs_dev)
evaluate("dev", W2, xs_dev, ys_dev, zs_dev)
evaluate("dev", W3, xs_dev, ys_dev, zs_dev)
evaluate("dev", W4, xs_dev, ys_dev, zs_dev)
evaluate("dev", W5, xs_dev, ys_dev, zs_dev)
evaluate("dev", W6, xs_dev, ys_dev, zs_dev)

print()
evaluate("test", W1, xs_test, ys_test, zs_test)
evaluate("test", W2, xs_test, ys_test, zs_test)
evaluate("test", W3, xs_test, ys_test, zs_test)
evaluate("test", W4, xs_test, ys_test, zs_test)
evaluate("test", W5, xs_test, ys_test, zs_test)
evaluate("test", W6, xs_test, ys_test, zs_test)

dev loss: 2.147305727005005
dev loss: 2.154637575149536
dev loss: 2.149310827255249
dev loss: 2.1454029083251953
dev loss: 2.1643178462982178
dev loss: 2.2275702953338623

test loss: 2.146620988845825
test loss: 2.1528115272521973
test loss: 2.1466989517211914
test loss: 2.146679162979126
test loss: 2.163553237915039
test loss: 2.226503610610962


**Best regularization strength for dev set loss: 0.01**
* Dev set loss: 2.1454029083251953
* Test set loss: 2.146679162979126

Observation: As regularization increases, train loss goes up while dev loss first improves then gets worse.

# E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

In [164]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [165]:
def initialize_set(word_set: list[str]):
    xs, ys = [], []
    for w in word_set:
      chs = ['.'] + list(w) + ['.']
      for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
    return torch.tensor(xs), torch.tensor(ys)

xs_train, ys_train = initialize_set(training_set)

In [168]:
final_loss = None
for k in range(100):
    ####################
    # One-hot encoding #
    ####################
    # xenc = F.one_hot(xs_train, num_classes=27).float()
    # logits = xenc @ W
    ##################
    #### Indexing ####
    ##################
    logits = W[xs_train, :]
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(xs_train.nelement()), ys_train].log().mean() + 0.01*(W**2).mean()
    final_loss = loss.item()
    
    W.grad = None
    loss.backward()

    W.data += -50 * W.grad
print(final_loss)

2.480376958847046


# E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

### Bigram

In [169]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [170]:
def initialize_set(word_set: list[str]):
    xs, ys = [], []
    for w in word_set:
      chs = ['.'] + list(w) + ['.']
      for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
    return torch.tensor(xs), torch.tensor(ys)

xs_train, ys_train = initialize_set(training_set)

In [171]:
final_loss = None
for k in range(100):
    logits = W[xs_train, :]
    ####################
    ## Manual softmax ##
    ####################
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(xs_train.nelement()), ys_train].log().mean() + 0.01*(W**2).mean()
    ###################
    ## Cross entropy ##
    ###################
    loss = F.cross_entropy(logits, ys_train)
    final_loss = loss.item()
    
    W.grad = None
    loss.backward()

    W.data += -50 * W.grad
print(final_loss)

2.471642255783081


### Trigram

In [174]:
def initialize_set(word_set: list[str]):
    xs, ys, zs = [], [], []
    for w in word_set:
      chs = ['.'] + list(w) + ['.']
      for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)
    return torch.tensor(xs), torch.tensor(ys), torch.tensor(zs)

xs_train, ys_train, zs_train = initialize_set(training_set)

In [175]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27, 27), generator=g, requires_grad=True)

In [177]:
final_loss = None
for k in range(100):
    logits = W[xs_train, ys_train, :]
    ####################
    ## Manual softmax ##
    ####################
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdim=True)
    # loss = -probs[torch.arange(xs_train.nelement()), zs_train].log().mean()
    ###################
    ## Cross entropy ##
    ###################
    loss = F.cross_entropy(logits, zs_train)
    final_loss = loss.item()
    
    W.grad = None
    loss.backward()
    
    # update
    W.data += -500 * W.grad

print(final_loss)

2.1058804988861084
